Advanced SQL analysis




4.1 Load the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import sqlite3

sales = pd.read_csv(
    "online_retail_clean_v2.csv",
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string"
    },
    low_memory=False
)

print("Shape:", sales.shape)
sales.head()

Shape: (524878, 12)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth,DayName,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8


In [ ]:
sql_sales = sales[[
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
    "Revenue",
    "YearMonth",
    "DayName",
    "Hour"
]].copy()

sql_sales.columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
    "revenue",
    "year_month",
    "day_name",
    "hour"
]

sql_sales["invoice_date"] = (
    sql_sales["invoice_date"]
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

sql_sales.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,revenue,year_month,day_name,hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8


In [ ]:
connection = sqlite3.connect("online_retail.db")

sql_sales.to_sql(
    "retail_sales",
    connection,
    if_exists="replace",
    index=False
)

connection.execute(
    "CREATE INDEX IF NOT EXISTS idx_invoice ON retail_sales(invoice_no)"
)

connection.execute(
    "CREATE INDEX IF NOT EXISTS idx_customer ON retail_sales(customer_id)"
)

connection.execute(
    "CREATE INDEX IF NOT EXISTS idx_product ON retail_sales(stock_code)"
)

connection.execute(
    "CREATE INDEX IF NOT EXISTS idx_month ON retail_sales(year_month)"
)

connection.commit()

print("SQLite database created successfully.")

SQLite database created successfully.


In [ ]:
def run_query(query):
    return pd.read_sql_query(query, connection)

Query 1: Main business KPIs

In [ ]:
query_1 = """
SELECT
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT stock_code) AS unique_products,
    SUM(quantity) AS units_sold,
    ROUND(
        SUM(revenue) / COUNT(DISTINCT invoice_no),
        2
    ) AS average_order_value
FROM retail_sales;
"""

run_query(query_1)

,total_revenue,total_orders,unique_customers,unique_products,units_sold,average_order_value
0,10642110.8,19960,4338,3922,5572420,533.17


Query 2: Monthly performance using GROUP BY

In [ ]:
query_2 = """
SELECT
    year_month,
    ROUND(SUM(revenue), 2) AS monthly_revenue,
    COUNT(DISTINCT invoice_no) AS orders,
    SUM(quantity) AS units_sold,
    ROUND(
        SUM(revenue) / COUNT(DISTINCT invoice_no),
        2
    ) AS average_order_value
FROM retail_sales
GROUP BY year_month
ORDER BY year_month;
"""

monthly_sql = run_query(query_2)
monthly_sql

,year_month,monthly_revenue,orders,units_sold,average_order_value
0,2010-12,821452.73,1559,358019,526.91
1,2011-01,689811.61,1086,387099,635.19
2,2011-02,522545.56,1100,282934,475.04
3,2011-03,716215.26,1454,376599,492.58
4,2011-04,536968.49,1246,307953,430.95
5,2011-05,769296.61,1681,395001,457.64
6,2011-06,760547.01,1533,388511,496.12
7,2011-07,718076.12,1475,399693,486.83
8,2011-08,757841.38,1361,421020,556.83
9,2011-09,1056435.19,1837,569573,575.09


Query 3: Customer value using a CTE and CASE WHEN

In [ ]:
query_3 = """
WITH customer_spending AS (
    SELECT
        customer_id,
        COUNT(DISTINCT invoice_no) AS orders,
        ROUND(SUM(revenue), 2) AS total_spending
    FROM retail_sales
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
),
customer_segments AS (
    SELECT
        customer_id,
        orders,
        total_spending,
        CASE
            WHEN total_spending >= 10000 THEN 'Very High Value'
            WHEN total_spending >= 5000 THEN 'High Value'
            WHEN total_spending >= 1000 THEN 'Medium Value'
            ELSE 'Low Value'
        END AS customer_segment
    FROM customer_spending
)
SELECT
    customer_segment,
    COUNT(*) AS number_of_customers,
    ROUND(AVG(total_spending), 2) AS average_spending,
    ROUND(SUM(total_spending), 2) AS segment_revenue
FROM customer_segments
GROUP BY customer_segment
ORDER BY segment_revenue DESC;
"""

run_query(query_3)

,customer_segment,number_of_customers,average_spending,segment_revenue
0,Very High Value,104,35097.02,3650089.97
1,Medium Value,1390,2152.11,2991432.26
2,High Value,170,6709.20,1140564.36
3,Low Value,2674,413.28,1105122.30


Query 4: Rank countries by revenue

In [ ]:
query_4 = """
WITH country_performance AS (
    SELECT
        country,
        ROUND(SUM(revenue), 2) AS total_revenue,
        COUNT(DISTINCT invoice_no) AS orders,
        COUNT(DISTINCT customer_id) AS customers
    FROM retail_sales
    GROUP BY country
)
SELECT
    country,
    total_revenue,
    orders,
    customers,
    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank
FROM country_performance
ORDER BY revenue_rank;
"""

run_query(query_4).head(15)

,country,total_revenue,orders,customers,revenue_rank
0,United Kingdom,9001744.09,18019,3920,1
1,Netherlands,285446.34,94,9,2
2,EIRE,283140.52,288,3,3
3,Germany,228678.40,457,94,4
4,France,209625.37,392,87,5
5,Australia,138453.81,57,9,6
6,Spain,61558.56,90,30,7
7,Switzerland,57067.60,54,21,8
8,Belgium,41196.34,98,25,9
9,Sweden,38367.83,36,8,10


Query 5: Top three products in each major market

In [ ]:
query_5 = """
WITH product_performance AS (
    SELECT
        country,
        stock_code,
        description,
        ROUND(SUM(revenue), 2) AS product_revenue,
        SUM(quantity) AS units_sold
    FROM retail_sales
    WHERE country IN (
        'United Kingdom',
        'Netherlands',
        'EIRE',
        'Germany',
        'France'
    )
    AND stock_code NOT IN (
        'POST', 'DOT', 'M', 'D', 'S',
        'AMAZONFEE', 'BANK CHARGES', 'CRUK'
    )
    GROUP BY country, stock_code, description
),
ranked_products AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY country
            ORDER BY product_revenue DESC
        ) AS product_rank
    FROM product_performance
)
SELECT *
FROM ranked_products
WHERE product_rank <= 3
ORDER BY country, product_rank;
"""

run_query(query_5)

,country,stock_code,description,product_revenue,units_sold,product_rank
0,EIRE,22423,REGENCY CAKESTAND 3 TIER,7793.25,679,1
1,EIRE,C2,CARRIAGE,5240.00,106,2
2,EIRE,22838,3 TIER CAKE TIN RED AND CREAM,4265.55,333,3
3,France,23084,RABBIT NIGHT LIGHT,7277.20,4024,1
4,France,22423,REGENCY CAKESTAND 3 TIER,2816.85,239,2
5,France,21731,RED TOADSTOOL LED NIGHT LIGHT,2169.75,1315,3
6,Germany,22423,REGENCY CAKESTAND 3 TIER,9061.95,809,1
7,Germany,22326,ROUND SNACK BOXES SET OF4 WOODLAND,3563.55,1221,2
8,Germany,22328,ROUND SNACK BOXES SET OF 4 FRUITS,1982.40,672,3
9,Netherlands,23084,RABBIT NIGHT LIGHT,9568.48,4801,1


Query 6: Monthly growth using LAG

In [ ]:
query_6 = """
WITH monthly_revenue AS (
    SELECT
        year_month,
        SUM(revenue) AS revenue
    FROM retail_sales
    GROUP BY year_month
),
monthly_comparison AS (
    SELECT
        year_month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY year_month
        ) AS previous_month_revenue
    FROM monthly_revenue
)
SELECT
    year_month,
    ROUND(revenue, 2) AS revenue,
    ROUND(previous_month_revenue, 2) AS previous_revenue,
    ROUND(
        (
            revenue - previous_month_revenue
        ) / NULLIF(previous_month_revenue, 0) * 100,
        2
    ) AS monthly_growth_percentage
FROM monthly_comparison
ORDER BY year_month;
"""

run_query(query_6)

,year_month,revenue,previous_revenue,monthly_growth_percentage
0,2010-12,821452.73,NaN,NaN
1,2011-01,689811.61,821452.73,-16.03
2,2011-02,522545.56,689811.61,-24.25
3,2011-03,716215.26,522545.56,37.06
4,2011-04,536968.49,716215.26,-25.03
5,2011-05,769296.61,536968.49,43.27
6,2011-06,760547.01,769296.61,-1.14
7,2011-07,718076.12,760547.01,-5.58
8,2011-08,757841.38,718076.12,5.54
9,2011-09,1056435.19,757841.38,39.40


Query 7: Running revenue using SUM() OVER()

In [ ]:
query_7 = """
WITH monthly_revenue AS (
    SELECT
        year_month,
        SUM(revenue) AS revenue
    FROM retail_sales
    GROUP BY year_month
)
SELECT
    year_month,
    ROUND(revenue, 2) AS monthly_revenue,
    ROUND(
        SUM(revenue) OVER (
            ORDER BY year_month
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ),
        2
    ) AS cumulative_revenue
FROM monthly_revenue
ORDER BY year_month;
"""

run_query(query_7)

,year_month,monthly_revenue,cumulative_revenue
0,2010-12,821452.73,821452.73
1,2011-01,689811.61,1511264.34
2,2011-02,522545.56,2033809.90
3,2011-03,716215.26,2750025.16
4,2011-04,536968.49,3286993.65
5,2011-05,769296.61,4056290.26
6,2011-06,760547.01,4816837.27
7,2011-07,718076.12,5534913.39
8,2011-08,757841.38,6292754.77
9,2011-09,1056435.19,7349189.96


Query 8: Customers spending above average

In [ ]:
query_8 = """
WITH customer_spending AS (
    SELECT
        customer_id,
        COUNT(DISTINCT invoice_no) AS orders,
        SUM(revenue) AS total_spending
    FROM retail_sales
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
)
SELECT
    customer_id,
    orders,
    ROUND(total_spending, 2) AS total_spending
FROM customer_spending
WHERE total_spending > (
    SELECT AVG(total_spending)
    FROM customer_spending
)
ORDER BY total_spending DESC;
"""

above_average_customers = run_query(query_8)

print(
    "Customers spending above average:",
    len(above_average_customers)
)

above_average_customers.head(20)

Customers spending above average: 871


,customer_id,orders,total_spending
0,14646.0,73,280206.02
1,18102.0,60,259657.30
2,17450.0,46,194390.79
3,16446.0,2,168472.50
4,14911.0,201,143711.17
5,12415.0,21,124914.53
6,14156.0,55,117210.08
7,17511.0,31,91062.38
8,16029.0,63,80850.84
9,12346.0,1,77183.60


Save the SQL results

In [ ]:
monthly_sql.to_csv(
    "sql_monthly_performance.csv",
    index=False
)

above_average_customers.to_csv(
    "sql_high_value_customers.csv",
    index=False
)

connection.close()

print("SQL analysis completed successfully.")

SQL analysis completed successfully.


In [ ]:
transaction_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

print("Rows:", len(sales))
print(
    "Duplicate transactions:",
    sales.duplicated(subset=transaction_columns).sum()
)

assert len(sales) == 524878
assert sales.duplicated(subset=transaction_columns).sum() == 0

print("Correct duplicate-free dataset confirmed.")

Rows: 524878
Duplicate transactions: 0
Correct duplicate-free dataset confirmed.


In [ ]:
cleaning_base = sales.drop_duplicates().copy()

# Add quality flags only after removing duplicates